# Detecting noisy monitors

This notebook shows how to detect noisy monitors in a dataset using the WhyLabs Monitor Diagnoser. It uses the diagnoser to automatically detect the noisiest monitor for dataset, get a diagnosis of
the conditions causing the noise, get recommended changes and where automatable, apply those changes.

## Install requirements

In [1]:
# %pip install whylabs-toolkit[diagnoser]

## Setup whylabs API connection

First, set up the information to connect to WhyLabs. Update the org_id, dataset_id and api_key in the following before running it.


In [2]:
import getpass
from whylabs_toolkit.monitor.diagnoser.helpers.utils import env_setup

org_id = 'org-0'
dataset_id = 'model-0'
api_key = getpass.getpass()
api_endpoint = 'https://songbird.development.whylabsdev.com'

env_setup(
    org_id=org_id,
    dataset_id=dataset_id,
    api_key=api_key,
    whylabs_endpoint=api_endpoint
)

TypeError: issubclass() arg 1 must be a class

Initialize the Monitor Diagnoser with the org_id and dataset_id.

In [ ]:
from whylabs_toolkit.monitor.diagnoser.monitor_diagnoser import MonitorDiagnoser
diagnoser = MonitorDiagnoser(org_id, dataset_id)

## Run the default diagnosis

With no further input, the diagnoser will make a series of calls to identify the noisiest monitor, segment and columns; and then perform a diagnosis.

In [ ]:
# for now, we need to enforce this to run using local server
import os
os.environ['USE_LOCAL_SERVER'] = 'server'
monitor_report = diagnoser.diagnose()
monitor_report

In [ ]:
print(monitor_report.describe())

The monitor report can be serialized to a JSON file for later use.

In [ ]:
with open('monitor_report.json', 'w') as f:
    f.write(monitor_report.json())

In [ ]:
from whylabs_toolkit.monitor.diagnoser.models import MonitorDiagnosisReport

with open('monitor_report.json', 'r') as f:
    monitor_report = MonitorDiagnosisReport.parse_raw(f.read())
print(monitor_report.json(indent=2))

## Ask for recommended changes

Given the diagnosis report for the monitor, the ChangeRecommender will recommend changes to make to the monitor. By default it will make recommendations for all columns where it has detected noise-related conditions. Set the `min_anomaly_count` property to restrict this to only columns that caused a certain number of anomalies.


In [ ]:
from whylabs_toolkit.monitor.diagnoser.recommendation.change_recommender import ChangeRecommender

recommender = ChangeRecommender(monitor_report)
recommender.min_anomaly_count = 1
changes = recommender.recommend()
print('\n'.join([f'{i+1}. {c.describe()}' for i, c in enumerate(changes)]))

## Execute automatable changes

A subset of recommended changes can be executed automatically by the recommender. Pass the ones you want to make into the `make_changes` call, or pass all changes if you want it to make all of the automatable changes.

In [ ]:
automatable_changes = [c for c in changes if c.can_automate()]
print('\n'.join([c.describe() for c in automatable_changes]))

In [ ]:
change_results = recommender.make_changes(automatable_changes)
print(change_results.describe())

Note that the monitor will still appear to the diagnoser as the noisiest monitor until enough time has passed for the impact of the monitor changes to be observed. You may want to use the WhyLabs preview UI to view what impacts may be expected from the change.

## Reviewing other noisy monitors

The diagnoser can be used to review other noisy monitors in the dataset. The `noisy_monitors` property will return a list of the noisiest monitors, and the `monitor_id_to_diagnose` property can be set to the monitor_id of the monitor to diagnose.

In [ ]:
import pandas as pd
noisy_monitors_df = pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors])
noisy_monitors_df

In [ ]:
diagnoser.monitor_id_to_diagnose = noisy_monitors_df.iloc[1]['monitor_id']
monitor_report = diagnoser.diagnose()
print(monitor_report.describe())

You can also use the `noisy_monitors_with_actions` property to prioritize noise in monitors with actions, as these are most likely to cause alert fatigue.

In [ ]:
pd.DataFrame.from_records([m.dict() for m in diagnoser.noisy_monitors_with_actions])
